In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# video_trajectory_payload_framewise_sync_m05_v1

Set CONFIG_PATH to a JSON based on `experiments/wan_state_clock/configs/video_trajectory_payload_framewise_sync_m05_v1.json` with explicit RGB path/SHA and hash-bound reporting files. Only the declared fixed geometry is accepted. Historical exact inputs are separately labeled in configs/historical. No implicit historical Drive source, new scientific claim, or automatic model execution by agents.

Fixed denominator: {'source': 1, 'condition': 1, 'sync_reads': 2, 'payload_reads': 2, 'final_bits': 64}. Process failures and all slots remain; wrong-key controls are descriptive, not FPR.

Published source: 9054f67337088f4e000ea8225c72d1ca18ce3a9c

In [ ]:
SOURCE_SHA = '9054f67337088f4e000ea8225c72d1ca18ce3a9c'
CONFIG_PATH = None  # Explicit user-supplied configuration file.
ENTRYPOINT = 'experiments.wan_state_clock.video_trajectory_payload_framewise_sync_m05_v1_run'
OUTPUT_ROOT = '/content/drive/MyDrive/Video-WM/video_trajectory_payload_framewise_sync_m05_v1'
FIXED = {'source': 1, 'condition': 1, 'sync_reads': 2, 'payload_reads': 2, 'final_bits': 64}
PINS = {'torch': '2.11.0', 'diffusers': '0.39.0', 'transformers': '4.57.6', 'numpy': '2.1.3', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2'}
REQUIRES_MEDIA = True
from pathlib import Path
import datetime,hashlib,json,os,signal,subprocess,sys,time,traceback
if SOURCE_SHA is None:
    raise RuntimeError('UNPUBLISHED_DRAFT: publish reviewed source and rebuild with its immutable SHA before Run all')
if not isinstance(SOURCE_SHA,str) or len(SOURCE_SHA)!=40:
    raise RuntimeError('immutable 40-character source SHA required')
if CONFIG_PATH is None or not Path(CONFIG_PATH).is_file():
    raise RuntimeError('Set CONFIG_PATH to an explicit input configuration JSON; no historical Drive input is implicit')
CONFIG_BYTES=Path(CONFIG_PATH).read_bytes()
CONFIG_SHA256=hashlib.sha256(CONFIG_BYTES).hexdigest()
INPUT_CONFIG=json.loads(CONFIG_BYTES)
if INPUT_CONFIG.get('name')!=ENTRYPOINT.rsplit('.',1)[-1].removesuffix('_run'):
    raise RuntimeError('configuration name does not belong to the fixed entrypoint')
if INPUT_CONFIG.get('fixed_denominator')!=FIXED:
    raise RuntimeError('configuration fixed denominator mismatch')
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path(OUTPUT_ROOT)/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-'+STAMP)
PYTHON=sys.executable
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    captured_traceback=''.join(traceback.format_exception(type(exc),exc,exc.__traceback__))
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=f'{type(exc).__name__}: {exc}',traceback=captured_traceback,fixed_denominator=FIXED))
def logged(command,stage,cwd=None,check=True):
    child=None;code=None;primary=None;primary_tb=None;cleanup_errors=[]
    def retain_cleanup_error(label,exc):
        nonlocal primary,primary_tb
        if primary is None and not isinstance(exc,Exception):
            primary=exc;primary_tb=exc.__traceback__
        else:
            cleanup_errors.append(label+': '+repr(exc))
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            popen_group={'start_new_session':True} if os.name=='posix' else {}
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,**popen_group)
            try:
                for line in child.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                code=child.wait()
            except BaseException as exc:
                primary=exc;primary_tb=exc.__traceback__
            finally:
                if child is not None:
                    if os.name=='posix':
                        pgid=child.pid
                        try:os.killpg(pgid,signal.SIGTERM)
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('group SIGTERM',exc)
                        try:
                            deadline=time.monotonic()+5.0
                            while time.monotonic()<deadline:
                                try:os.killpg(pgid,0)
                                except ProcessLookupError:break
                                except BaseException as exc:
                                    retain_cleanup_error('group probe',exc);break
                                time.sleep(0.05)
                        except BaseException as exc:
                            retain_cleanup_error('group TERM grace',exc)
                        finally:
                            try:os.killpg(pgid,signal.SIGKILL)
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('group SIGKILL',exc)
                    elif child.poll() is None:
                        try:child.terminate()
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('terminate',exc)
                    try:
                        try:code=child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            try:child.kill()
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('kill',exc)
                            try:code=child.wait(timeout=10)
                            except BaseException as exc:retain_cleanup_error('wait after kill',exc)
                        except BaseException as exc:retain_cleanup_error('wait',exc)
                    finally:
                        if child.stdout is not None:
                            try:child.stdout.close()
                            except BaseException as exc:retain_cleanup_error('stdout close',exc)
            if primary is None and not cleanup_errors:
                log.write('EXIT '+str(code)+'\n');log.flush()
    except BaseException as wrapper_error:
        if primary is None:
            primary=wrapper_error;primary_tb=wrapper_error.__traceback__
        elif wrapper_error is not primary:
            cleanup_errors.append('log wrapper/close: '+repr(wrapper_error))
    if primary is None and cleanup_errors:
        primary=RuntimeError('process cleanup failed: '+'; '.join(cleanup_errors))
        primary_tb=primary.__traceback__
    if primary is None and check and code:
        primary=subprocess.CalledProcessError(code,command)
        primary_tb=primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary,'add_note'):
            primary.add_note('secondary cleanup errors: '+'; '.join(cleanup_errors))
        try:failed(stage,primary)
        except BaseException as record_error:
            if hasattr(primary,'add_note'):primary.add_note('failure record error: '+repr(record_error))
        raise primary.with_traceback(primary_tb)
    return code
write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,config_sha256=CONFIG_SHA256,fixed_denominator=FIXED))


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    if REQUIRES_MEDIA:
        import shutil
        if any(shutil.which(n) is None for n in ('ffmpeg','ffprobe')):
            logged(['apt-get','update'],'MEDIA_TOOL_UPDATE',check=False)
            logged(['apt-get','install','-y','ffmpeg'],'MEDIA_TOOL_REPAIR')
    probe="import importlib.metadata as m; p="+repr(PINS)+"; a={k:m.version(k) for k in p}; assert all(a[k].split('+')[0]==v for k,v in p.items()),a; from diffusers import AutoencoderKL,AutoencoderKLWan; print(a)"
    if logged([PYTHON,'-c',probe],'DEPENDENCY_PROBE',check=False):
        logged([PYTHON,'-m','pip','install',*[k+'=='+v for k,v in PINS.items()]],'DEPENDENCY_REPAIR')
        logged([PYTHON,'-c',probe],'DEPENDENCY_REPROBE')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_COMPLETE',source_sha=actual,fixed_denominator=FIXED,dependency_check_returncode=dependency_code))
except Exception as exc:
    try:failed('SOURCE_OR_ENVIRONMENT',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:pass
finally:token=None
command=[PYTHON,'-u','-m',ENTRYPOINT,'--config',str(CONFIG_PATH),'--output',str(RUN_OUTPUT)]
try:
    if hashlib.sha256(Path(CONFIG_PATH).read_bytes()).hexdigest()!=CONFIG_SHA256:raise RuntimeError('configuration changed after setup')
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,config_sha256=CONFIG_SHA256,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; fixed denominator retained')
except Exception as exc:
    try:failed('RUNNER',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA:raise RuntimeError('source identity mismatch')
if result['config_sha256']!=CONFIG_SHA256:raise RuntimeError('result configuration identity mismatch')
if result['fixed_denominator']!=FIXED:raise RuntimeError('result fixed denominator mismatch')
print('Status:',result['status'],'Counts:',result.get('counts'),'Calls:',result.get('calls'))
print('Actual config SHA:',result['config_sha256'],'Fixed denominator:',result['fixed_denominator'])
for sid,row in result.get('estimates',{}).items():
    print('Estimate',sid,row)
for sid,row in result.get('sync_posthoc',result.get('path_posthoc',{})).items():
    print('Sync/path posthoc',sid,row)
for lid,row in result.get('payload_posthoc',result.get('posthoc',{})).items():
    margins=[x['signed_normalized_margin'] for x in row.get('bit_rows',[]) if 'signed_normalized_margin' in x]
    print('Payload',lid,'key=',row.get('key_role'),'errors=',row.get('bit_errors'),'error_bits=',row.get('error_bits'),'minmargin=',min(margins,default=None),'alias=',row.get('alias_of'))
print('Failures:',result['failures'])
print('Result:',RESULT_PATH)
print('Execution evidence only; aliases are not independent observations; full readouts remain in result and sidecars.')
